### data assessment

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
data_path = Path('../data/raw')

In [5]:
df_user = pd.read_csv(data_path/'user.csv')
df_card = pd.read_csv(data_path/'card.csv')
df_mcc = pd.read_csv(data_path/'mcc.csv')
df_transaction = pd.read_csv(data_path/'transaction.csv')

In [6]:
for df in [df_user, df_card, df_mcc, df_transaction]:
    print(df.info())
    print(df.describe())
    print("\n")

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   client_id          2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   str    
 6   address            2000 non-null   str    
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   str    
 10  yearly_income      2000 non-null   str    
 11  total_debt         2000 non-null   str    
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 218.9 KB
None
         client_id  current_age  retirement_age   birth_year  birth_mon

### Note

`df_user`

- no missing values
- `per_capita_income` and `yearly_income` and `total_debt` need to remove dollar signs "$"
- check if `latitude` and `longitude` combinations have duplicates but `address` is different

In [23]:
location_address_duplicates = (
    df_user.groupby(['latitude', 'longitude'])['address']
    .nunique()
    .reset_index(name='address_count')
    .query('address_count > 1')
)

location_address_duplicates

,latitude,longitude,address_count
11,25.77,-80.20,12
12,25.92,-97.48,2
13,26.11,-80.39,2
15,26.14,-81.79,2
16,26.14,-80.13,8
...,...,...,...
1510,47.54,-122.58,2
1512,47.61,-122.30,2
1515,47.67,-122.18,2
1521,47.79,-122.20,2


`df_card`

- no missing values
- check if `year_pin_last_changed` < `acct_open_date`

In [20]:
# Compare the PIN-change year with the account opening year
pin_changed_year = pd.to_numeric(df_card['year_pin_last_changed'], errors='coerce')
acct_open_year = pd.to_datetime(df_card['acct_open_date'], errors='coerce').dt.year

invalid_pin_dates = df_card[pin_changed_year < acct_open_year]

print(f'Rows where year_pin_last_changed < acct_open_date: {len(invalid_pin_dates)}')

Rows where year_pin_last_changed < acct_open_date: 0


`df_mcc`

- no missing values
- check duplicates in `mcc_id`

In [25]:
mcc_duplicates = df_mcc[df_mcc['mcc_id'].duplicated(keep=False)].sort_values('mcc_id')

print(f"Duplicate mcc_id values: {mcc_duplicates['mcc_id'].nunique()}")

Duplicate mcc_id values: 0


`df_transaction`

- missing values in `merchant_state` and `zip` and `errors`
- 303 distinct `client_id` < 2000 existing clients
- if `merchant_city` == "ONLINE" then `merchant_state` and `zip` have missing values

In [26]:
transaction_client_ids = set(df_transaction['client_id'].dropna())
user_client_ids = set(df_user['client_id'].dropna())

transaction_not_in_users = transaction_client_ids - user_client_ids
users_without_transactions = user_client_ids - transaction_client_ids

print(f"Transaction client_ids not found in df_user: {len(transaction_not_in_users)}")
print(sorted(transaction_not_in_users))

print(f"User client_ids without transactions: {len(users_without_transactions)}")

Transaction client_ids not found in df_user: 0
[]
User client_ids without transactions: 1697
